<a href="https://colab.research.google.com/github/httpsfahim/practice-projects/blob/main/ResearchDaily.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U google-genai

import requests
import time
import google.generativeai as genai
from google.colab import userdata

# Load API keys
gemini_api_key = userdata.get('GEMINI_API')
ss_api_key = userdata.get('SEMANTIC_SCHOLAR_API')
genai.configure(api_key=gemini_api_key)

model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')


def get_user_input():
    while True:
        query = input('Enter a research topic: ').strip()
        if query:
            break
        print('Please enter a valid topic.', flush=True) # Added flush=True

    while True:
        try:
            limit = int(input('How many papers do you want (1-10)? '))
            if 1 <= limit <= 10:
                break
            else:
                print('Enter a number between 1 and 10.', flush=True)
        except:
            print('Invalid number.', flush=True)

    # Use a single print for the menu to ensure it appears at once
    menu = "\nChoose a tone preset:\n1. professional\n2. conversational\n3. bold\n4. creative\n"
    print(menu, flush=True)

    while True:
        choice = input('Enter 1, 2, 3, or 4: ').strip()
        if choice in ['1', '2', '3', '4']:
            break
        print('Invalid choice. Try again.', flush=True)

    return query, limit, choice


def get_tone_and_temperature(choice):
    settings = {
        '1': ('professional', 0.3),
        '2': ('conversational', 0.6),
        '3': ('bold', 0.9),
        '4': ('creative', 1.0)  # Fixed: max is 1.0
    }
    return settings[choice]


def fetch_papers(query, limit, retries=3, delay=5):
    url = 'https://api.semanticscholar.org/graph/v1/paper/search'
    params = {
        'query': query,
        'limit': limit,
        'fields': 'title,authors,year,abstract',  # Removed broken sort param
    }
    headers = {'x-api-key': ss_api_key}

    for attempt in range(retries):
        response = requests.get(url, params=params, headers=headers)
        if response.status_code == 200:
            return response
        elif response.status_code == 429:
            wait = delay * (attempt + 1)  # 5s, 10s, 15s
            print(f'Rate limited. Waiting {wait} seconds before retrying...')
            time.sleep(wait)
        else:
            print(f'Unexpected error: {response.status_code}')
            return response

    return response


def generate_summary(title, abstract, tone, temperature):
    if not abstract:
        return 'No abstract available.'

    tone_map = {
        'professional': 'Use a formal academic tone.',
        'conversational': 'Explain clearly in a friendly way.',
        'bold': 'Be assertive and highlight key insights.',
        'creative': 'Use creative and engaging perspectives.'
    }

    prompt = f"""
{tone_map[tone]}

Summarize this research paper in at least 150-200 words, making sure to cover all key points including the purpose, methodology, findings, and conclusions:

Title: {title}
Abstract: {abstract}
"""

    try:
        time.sleep(1)  # Avoid Gemini rate limits
        response = model.generate_content(
            prompt,
            generation_config={
                'temperature': temperature,
                'max_output_tokens': 600
            }
        )
        return response.text
    except Exception as e:
        return f'Error generating summary: {e}'


def display_results(data, tone, temperature, requested_limit):
    papers = data.get('data', [])

    # Filter out papers with no abstract
    papers = [p for p in papers if p.get('abstract')]

    if not papers:
        print('Sorry for the inconvenience, not enough papers found with abstracts.')
        return False

    if len(papers) < requested_limit:
        print(f'Sorry for the inconvenience, only {len(papers)} paper(s) found with abstracts instead of {requested_limit}.\n')

    for i, paper in enumerate(papers, start=1):
        title = paper.get('title', 'No title')
        authors = [a.get('name', 'Unknown') for a in paper.get('authors', [])]
        year = paper.get('year', 'N/A')
        abstract = paper.get('abstract')

        print(f'\n{i}. {title}')
        print(f'Authors: {", ".join(authors)}')
        print(f'Year: {year}\n')

        summary = generate_summary(title, abstract, tone, temperature)
        print(summary)
        print('-' * 80)

    return True


# MAIN LOOP
while True:
    query, limit, choice = get_user_input()
    tone, temperature = get_tone_and_temperature(choice)

    response = fetch_papers(query, limit)

    if response.status_code == 200:
        data = response.json()
        success = display_results(data, tone, temperature, limit)
        if success:
            break
    else:
        print('Error:', response.status_code)